# Vanilla CNN training and inference

Run `preprocessing.ipynb` completely before running this notebook. This notebook loads the saved images, trains a CNN, evaluates it, and creates test predictions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Define folders and training settings.
project_folder = Path.cwd()
image_dataset_folder = project_folder / 'Image_dataset'
train_folder = image_dataset_folder / 'train'
validation_folder = image_dataset_folder / 'validation'
test_folder = image_dataset_folder / 'test' / 'unlabelled'
model_path = project_folder / 'vanilla_cnn.keras'
prediction_path = project_folder / 'test_predictions.csv'

image_height = 128
image_width = 128
number_of_classes = 50
batch_size = 32
number_of_epochs = 25
random_seed = 42

if not train_folder.exists() or not validation_folder.exists():
    raise FileNotFoundError('Run preprocessing.ipynb before this notebook.')


In [ ]:
# Load labelled images from the class folders.
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_folder,
    labels='inferred',
    label_mode='int',
    color_mode='grayscale',
    image_size=(image_height, image_width),
    batch_size=batch_size,
    shuffle=True,
    seed=random_seed
)

class_names = train_dataset.class_names

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_folder,
    labels='inferred',
    label_mode='int',
    color_mode='grayscale',
    image_size=(image_height, image_width),
    batch_size=batch_size,
    shuffle=False,
    class_names=class_names
)

print('Classes:', class_names)


In [ ]:
def build_vanilla_cnn():
    # The Rescaling layer changes pixels from 0-255 to 0-1.
    model = tf.keras.Sequential(name='vanilla_environmental_sound_cnn')
    model.add(tf.keras.layers.Input(shape=(image_height, image_width, 1)))
    model.add(tf.keras.layers.Rescaling(1.0 / 255.0))

    # First convolution block learns simple local image patterns.
    model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))

    # Second block learns more complex combinations of patterns.
    model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))

    # Dense layers perform the final classification.
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dropout(0.40))
    model.add(tf.keras.layers.Dense(number_of_classes, activation='softmax'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_vanilla_cnn()
model.summary()


In [ ]:
# Train the model and keep the best validation weights.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

training_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=number_of_epochs,
    callbacks=[early_stopping]
)


In [ ]:
# Plot accuracy and loss during training.
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(training_history.history['accuracy'], label='Training accuracy')
axes[0].plot(training_history.history['val_accuracy'], label='Validation accuracy')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(training_history.history['loss'], label='Training loss')
axes[1].plot(training_history.history['val_loss'], label='Validation loss')
axes[1].set_title('Loss'); axes[1].legend()
plt.show()


In [ ]:
# Calculate a full classification report.
true_labels = []
predicted_labels = []

for image_batch, label_batch in validation_dataset:
    probabilities = model.predict(image_batch, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    true_labels.extend(label_batch.numpy())
    predicted_labels.extend(predictions)

validation_accuracy = accuracy_score(true_labels, predicted_labels)
print(f'Validation accuracy: {validation_accuracy:.4f}')
print(classification_report(
    true_labels,
    predicted_labels,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
) )


In [ ]:
# Display the confusion matrix.
matrix = confusion_matrix(true_labels, predicted_labels, labels=np.arange(number_of_classes))
figure, axis = plt.subplots(figsize=(14, 14))
matrix_display = ConfusionMatrixDisplay(matrix, display_labels=class_names)
matrix_display.plot(ax=axis, xticks_rotation='vertical', colorbar=False)
plt.tight_layout()
plt.show()


In [ ]:
# Save the trained model.
model.save(model_path)
print('Saved model to:', model_path)


In [ ]:
# Predict the saved test images.
test_image_paths = sorted(test_folder.glob('*.png'))
prediction_rows = []

for image_path in test_image_paths:
    image = Image.open(image_path).convert('L')
    image_array = np.asarray(image, dtype=np.float32)
    image_array = image_array[np.newaxis, :, :, np.newaxis]
    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    prediction_rows.append({
        'filename': image_path.stem + '.wav',
        'target': predicted_index,
        'class_name': class_names[predicted_index],
        'confidence': float(probabilities[predicted_index])
    })

test_predictions = pd.DataFrame(prediction_rows)
test_predictions.to_csv(prediction_path, index=False)
display(test_predictions.head())
print('Saved predictions to:', prediction_path)


## Inference function

The next cell defines a reusable function. Give it the path of a real WAV file to receive the predicted class and confidence.

In [ ]:
def predict_audio_file(audio_path):
    # Import the same audio-processing library used during preprocessing.
    import librosa

    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f'Audio file not found: {audio_path}')

    audio_signal, _ = librosa.load(audio_path, sr=22050, mono=True)
    expected_samples = 22050 * 5
    if len(audio_signal) < expected_samples:
        audio_signal = np.pad(audio_signal, (0, expected_samples - len(audio_signal)))
    if len(audio_signal) > expected_samples:
        audio_signal = audio_signal[:expected_samples]

    mel_spectrogram = librosa.feature.melspectrogram(
        y=audio_signal, sr=22050, n_fft=1024, hop_length=512, n_mels=128, power=2.0
    )
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
    minimum_value = log_mel_spectrogram.min()
    maximum_value = log_mel_spectrogram.max()
    value_range = maximum_value - minimum_value
    if value_range == 0:
        value_range = 1
    image_values = ((log_mel_spectrogram - minimum_value) / value_range * 255).astype(np.uint8)
    image = Image.fromarray(image_values, mode='L').resize((128, 128))
    image_array = np.asarray(image, dtype=np.float32)
    image_array = image_array[np.newaxis, :, :, np.newaxis]

    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    return class_names[predicted_index], float(probabilities[predicted_index])
